In [ ]:
from torch.utils.data import Dataset, DataLoader  # as duas classes base pra organizar dados no torch
import torch
import torch.nn as nn

In [3]:
x = [[1,2],[3,4],[5,6],[7,8]]
y = [[3],[7],[11],[15]]

In [5]:
X = torch.tensor(x).float()
Y = torch.tensor(y).float()

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
X = X.to(device)
Y = Y.to(device)

In [ ]:
class MyDataset(Dataset):  # Dataset customizado, so precisa implementar __init__, __len__ e __getitem__
    def __init__(self,x,y):
        self.x = torch.tensor(x).float().to(device)
        self.y = torch.tensor(y).float().to(device)
    def __len__(self):
        # DataLoader usa isso pra saber quantos batches vai gerar
        return len(self.x)
    def __getitem__(self,ix):
        # isso pra pegar 1 amostra por vez
        return self.x[ix], self.y[ix]

In [9]:
ds = MyDataset(x,y)

In [ ]:
dl = DataLoader(ds, batch_size=2, shuffle=True)  # DataLoader cuida de agrupar em batches e embaralhar, parecido com o que o .fit do keras faz por baixo dos panos

In [ ]:
for x,y in dl:
    print(x,y)  # da pra iterar o dataloader direto num for, cada iteração é um batch

tensor([[3., 4.],
        [7., 8.]]) tensor([[ 7.],
        [15.]])
tensor([[5., 6.],
        [1., 2.]]) tensor([[11.],
        [ 3.]])


In [ ]:
import torch
import torch.nn as nn
from torch.optim import SGD  # otimizadores ficam em torch.optim, equivale ao tf.keras.optimizers

In [15]:
class MyNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 8)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(8,1)

    def forward(self,x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

In [ ]:
model = MyNeuralNet()
loss_func = nn.MSELoss()
# o otimizador recebe os parâmetros do modelo pra saber o que atualizar
opt = SGD(model.parameters(), lr = 0.001)

In [ ]:
losses = []
for _ in range(50):# 50 épocas
    for data in dl: 
        # pra cada batch do dataloader
        opt.zero_grad() # zera os gradientes acumulados
        x1, y1 = data
        loss_value = loss_func(model(x1), y1)
        loss_value.backward()  # backpropagation calcula o gradiente de tudo que participou dessa conta

        opt.step()              # aplica o passo de atualização dos pesos usando os gradientes calculados
        losses.append(loss_value.detach().numpy())  # detach() tira do grafo de gradientes antes de converter pra numpy